# Phi-3 Mini Internal Explained

In this notebook, we analyze the internal workings of the Phi-3 Mini model by Microsoft (microsoft/Phi-3-mini-4k-instruct). 

We'll explore model structure, inputs/outputs, attention weights, hidden states, and LM head predictions.

This complements Chapter 3 of "Hands-On Large Language Models" by Jay Alammar and Maarten Grootendorst,
with extended insights and extra debug-level inspection.

## 1. MPS Memory Cleanup for Apple Silicon (macOS)
Before loading a new model, we ensure memory is cleared (especially useful when using Apple's Metal backend).

In [6]:
def cleanup_mps_memory():
    """
    Frees MPS memory by deleting global variables 'model' and 'tokenizer' if they exist.
    Useful when you want to avoid passing model/tokenizer manually.
    """
    import gc
    import torch

    for var in ['model', 'tokenizer']:
        if var in globals():
            print(f"🔹 Deleting: {var}")
            del globals()[var]

    gc.collect()
    torch.mps.empty_cache()
    print("MPS memory cleaned.")
cleanup_mps_memory()

🔹 Deleting: model
🔹 Deleting: tokenizer
MPS memory cleaned.


## 2. Load the Phi-3 Mini Model and Tokenizer

In [7]:
import torch 
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [8]:
%%time
# Set device to Apple MPS if available, fallback to CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# Load tokenizer and model from Hugging Face
tokenizer = AutoTokenizer.from_pretrained('microsoft/Phi-3-mini-4k-instruct')
model = AutoModelForCausalLM.from_pretrained(
    'microsoft/Phi-3-mini-4k-instruct',
    torch_dtype=torch.float32,
    trust_remote_code=True
).to(device)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.
Loading checkpoint shards: 100%|██████████████████| 2/2 [00:07<00:00,  3.83s/it]


CPU times: user 3.78 s, sys: 8.23 s, total: 12 s
Wall time: 25.1 s


In [9]:
context_length = model.config.max_position_embeddings
print("Context length:", context_length)

Context length: 4096


📘 What is context length?

The context length is the maximum number of tokens that can be fed into the model at any step, including:
- The prompt
- Any previously generated tokens
- Padding tokens (if applicable)

For decoder-only models like Phi-3, each generation step sees the full prompt + all prior outputs.
So once the prompt + generated tokens hit the context limit, the model must:
- Either stop generating,
- Or truncate the oldest tokens (left-shifting a window), but only if allowed (like sliding window decoding)

🧠 Analogy:
Think of the model as having a whiteboard that fits 4096 tokens.
KV caching lets it avoid rewriting the entire board—just append the next word.
But once full, you either erase the beginning or stop writing.

## 3. Create a Pipeline for Text Generation
The model is decoder-only. We'll use a text generation pipeline to test a basic prompt.

In [10]:
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=50,
    do_sample=False
)

In [11]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."
output = generator(prompt)
print(output[0]['generated_text'])

You are not running the flash-attention implementation, expect numerical differences.


 Mention that you've taken steps to prevent it in the future.


Email to Sarah:

Subject: Sincere Apologies for the Gardening Mishap


Dear Sarah,


I hope


## 4. Manually Tokenize a Prompt
(Let's dig into the components of the Decoder)

In [12]:
prompt = 'The capital of Germany is'
input_ids = tokenizer(prompt, return_tensors='pt').input_ids

In [13]:
input_ids = input_ids.to(device)

In [14]:
input_ids

tensor([[ 450, 7483,  310, 9556,  338]], device='mps:0')

In [15]:
len(input_ids[0]) # 5 tokens in the input sequence

5

In [16]:
for i in input_ids[0]:
    print(tokenizer.decode(i))

The
capital
of
Germany
is


## 5. Inspect Model Architecture

In [17]:
model

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
          (rotary_emb): Phi3RotaryEmbedding()
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLU()
        )
        (input_layernorm): Phi3RMSNorm()
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
        (post_attention_layernorm): Phi3RMSNorm()
      )
    )
    (norm): Phi3RMSNorm()
  )
  (lm_head): Linear(in_features=3072, out_features=3206

### 🔍 5.1 Detailed Explanation of Model Architecture (`print(model)`)

The model is composed of **two main components**:

- **`model`**: The Transformer body — includes embeddings, decoder layers, normalization.
- **`lm_head`**: Final linear layer that maps hidden states to vocabulary logits.

---

#### 🧱 Embedding Layer

**`(embed_tokens): Embedding(32064, 3072)`**  
- Token embedding layer: maps input token IDs to dense vectors.
- **32064** = vocabulary size  
- **3072** = embedding size (model dimension)  
- Each token is represented as a learned 3072-dimensional vector.

---

#### ☂️ Dropout Layer

**`(embed_dropout): Dropout(p=0.0)`**  
- Dropout is a regularization method to prevent overfitting during training.
- Here, it's set to `0.0`, so **disabled** (as is typical during inference).

---

#### 🧠 Transformer Backbone

**`(layers): ModuleList(...)`**  
- Stack of **32 `Phi3DecoderLayer`** blocks.
- Each decoder block contains:
  - **Self-attention sub-layer**
  - **Feed-forward network (MLP)**
  - **Layer normalization (RMSNorm)**
  - **Residual connections**
  - **Dropout (training only)**

---

#### 🔁 Self-Attention Mechanism

**`(self_attn): Phi3Attention`**

- Implements **masked self-attention** — tokens can only attend to themselves and earlier tokens (causal).
- Key subcomponents:

  - **`qkv_proj`: Linear(3072 → 9216)**  
    - Fused linear layer that projects a token's embedding into **Query (Q), Key (K), and Value (V)** vectors.  
    - Why 9216? → 3072 × 3 = 9216 (Q, K, and V are each 3072-dim).
    - These are split and reshaped into:
      - **32 heads** × **96 dimensions** per head.

  - **`rotary_emb`: Phi3RotaryEmbedding**  
    - Injects positional information via **rotary position encodings (RoPE)**.
    - No separate learned position embeddings — instead, the attention mechanism itself incorporates position.

  - **`o_proj`: Linear(3072 → 3072)**  
    - Projects concatenated multi-head output back to the model dimension (3072).

---

#### 🧮 Feed-Forward (MLP) Block

**`(mlp): Phi3MLP`**

- Dense layers that expand and transform the token representation.
- Subcomponents:

  - **`gate_up_proj`: Linear(3072 → 16384)`**  
    - Expands the representation dramatically (over 5x).
    - Often used in a gated activation format (like Gated Linear Units).

  - **`down_proj`: Linear(8192 → 3072)`**  
    - Reduces the intermediate output back to model dimension.
    - Note: The use of 8192 here may suggest parallel paths or compression.

  - **`activation_fn`: SiLU (Swish)**  
    - Activation function: `SiLU(x) = x * sigmoid(x)`
    - Smooth, non-linear, and commonly used in modern LLMs.

---

#### 🧽 Normalization

- **`Phi3RMSNorm`** is used instead of traditional LayerNorm.
- Root Mean Square Normalization:
  - Normalizes based on the RMS of elements.
  - No learned bias term.
  - Often more numerically stable and faster.

- Applied at:
  - **`input_layernorm`**: Before attention
  - **`post_attention_layernorm`**: Before MLP
  - **`norm`**: After all decoder layers

---

#### 💬 Dropout (Training Only)

- **`resid_attn_dropout`** and **`resid_mlp_dropout`**:
  - Dropout applied after attention and MLP.
  - `p=0.0` in this config — so **inactive** during inference.

---

#### 📤 LM Head

**`(lm_head): Linear(3072 → 32064)`**  
- Final linear layer mapping from hidden state to vocabulary logits.
- For each token position, outputs a **vector of size = vocab_size (32064)**.
- During generation, we only care about the **logits of the last token**, used to predict the next word.

---

#### ✅ Summary

- Phi-3 Mini uses a **decoder-only transformer** layout, optimized for performance and efficiency.
- Modern features:
  - **Fused QKV projection**
  - **Rotary position embeddings**
  - **Large MLP expansions**
  - **RMSNorm normalization**
- Suited for **efficient inference** on edge devices, including Apple Silicon.

---

## 6. Run Forward Pass and extract internals (with model.model) 

In [19]:
model_output = model.model(
    input_ids,
    output_hidden_states=True,
    output_attentions=True
)

In [20]:
# Output Keys
print(model_output.__dict__.keys())

dict_keys(['last_hidden_state', 'past_key_values', 'hidden_states', 'attentions'])


### 6.1. last_hidden_state

In [73]:
# The last hidden state is the output of the final decoder layer.
# This tensor contains contextualized embeddings for each token in the input sequence.
# It's the input to the LM head, which maps these embeddings to vocabulary logits.

last_hidden = model_output.last_hidden_state

print("Last Hidden State Shape:", last_hidden.shape)
print("Format → [batch_size=1, sequence_length=5, hidden_dim=3072]")

# Interpretation:
# - 1 input sequence (batch size)
# - 5 tokens in the sequence (e.g., "The capital of Germany is")
# - Each token is represented by a 3072-dimensional vector (hidden size of Phi-3 Mini)

Last Hidden State Shape: torch.Size([1, 5, 3072])
Format → [batch_size=1, sequence_length=5, hidden_dim=3072]


In [21]:
model_output.last_hidden_state

tensor([[[-0.2650,  1.2063,  0.4180,  ..., -0.3720,  0.7845,  0.1119],
         [-0.1311,  0.3082,  0.3648,  ...,  0.5546, -0.1395, -0.5973],
         [-0.5876,  1.0704,  1.5502,  ..., -0.4205,  0.2827,  0.4111],
         [-0.4459,  0.8368,  0.2083,  ...,  0.0643,  0.1239, -0.1316],
         [-0.9913, -0.3077,  0.3185,  ...,  0.6040,  0.6447, -0.8183]]],
       device='mps:0', grad_fn=<MulBackward0>)

### 6.2. past_key_values (for KV caching)

In [71]:
# During generation, the model caches the key and value tensors from each self-attention layer.
# This avoids recomputing attention over the full sequence at every step, enabling fast autoregressive decoding.

print("Past KV cache — number of decoder layers:", len(model_output.past_key_values))

# Each decoder layer's cache is a tuple: (keys, values)
# Shape of each: [batch_size, num_heads, sequence_length, head_dim]
print("Each layer returns a tuple of (cached keys, cached values):", len(model_output.past_key_values[0]))

# Let's inspect one layer's KV shapes (e.g., layer 0)
kv_keys_shape = model_output.past_key_values[0][0].shape
kv_values_shape = model_output.past_key_values[0][1].shape

print("\nCached Keys Shape (Layer 0):")
print("Format → [batch_size=1, num_heads=32, seq_len=5, head_dim=96]")
print(kv_keys_shape)

print("\nCached Values Shape (Layer 0):")
print("Format → [batch_size=1, num_heads=32, seq_len=5, head_dim=96]")
print(kv_values_shape)

Past KV cache — number of decoder layers: 32
Each layer returns a tuple of (cached keys, cached values): 2

Cached Keys Shape (Layer 0):
Format → [batch_size=1, num_heads=32, seq_len=5, head_dim=96]
torch.Size([1, 32, 5, 96])

Cached Values Shape (Layer 0):
Format → [batch_size=1, num_heads=32, seq_len=5, head_dim=96]
torch.Size([1, 32, 5, 96])


In [27]:
# 1st dec block, cached values for 1st input token, 1st head 
model_output.past_key_values[0][1][0][0][0] 

tensor([ 1.7944e-03,  1.1460e-03,  1.6670e-03, -4.6573e-03, -4.4691e-03,
        -4.0361e-03, -7.9229e-03, -1.3515e-03,  7.4669e-04,  1.5295e-04,
         6.1482e-03,  4.6329e-03, -4.9143e-04,  1.1170e-03, -1.3717e-02,
         1.1629e-04,  1.2481e-02,  6.0976e-03,  1.7094e-03, -8.3313e-03,
         8.1427e-03, -9.9037e-03, -4.9646e-05, -4.9557e-03,  3.0852e-03,
        -1.1908e-03, -2.1121e-03, -2.2041e-03, -1.0028e-04, -1.0126e-03,
        -2.4793e-03, -6.0938e-03, -4.8202e-04,  2.5243e-03,  1.2461e-02,
        -2.5259e-03,  8.6835e-03,  1.1030e-02,  3.0970e-03,  6.3453e-03,
         1.6713e-02,  4.5255e-03, -2.1354e-03,  6.0450e-03, -1.2542e-02,
         6.4853e-03,  1.8143e-03,  3.5244e-03, -1.4630e-03, -6.6364e-05,
        -4.3673e-03,  4.3101e-03,  1.8741e-03, -1.1336e-03,  5.6008e-03,
         4.0665e-02, -6.1451e-03,  3.7116e-03, -5.6553e-03,  9.1595e-03,
         3.2279e-03, -8.1195e-03,  2.3177e-03,  2.4106e-03,  5.1838e-03,
         6.4173e-03, -5.2042e-03,  3.9384e-03, -5.6

In [28]:
# length of 1st dec block, cached values for 1st input token, 1st head 
len(model_output.past_key_values[0][1][0][0][0])

96

### 6.3. hidden_states

In [75]:
# The model outputs hidden states from:
# - The initial embedding layer (input token + position embeddings)
# - All 32 decoder layers
# So in total, we receive 33 hidden state tensors.

total_hidden_states = len(model_output.hidden_states)
embedding_output_shape = model_output.hidden_states[0].shape
final_layer_output_shape = model_output.hidden_states[-1].shape

print("Hidden States: Total layers (including embeddings):", total_hidden_states)
print("Embedding Layer Output Shape:", embedding_output_shape)
print("Final Decoder Layer Output Shape:", final_layer_output_shape)

# Each hidden state tensor has the shape: [batch_size=1, sequence_length=5, hidden_dim=3072]
# These represent the token representations as they evolve through the model:
# - The first (index 0) is after the embedding layer (includes token + positional embeddings)
# - The last (index -1) is the final contextualized representation before the LM head

Hidden States: Total layers (including embeddings): 33
Embedding Layer Output Shape: torch.Size([1, 5, 3072])
Final Decoder Layer Output Shape: torch.Size([1, 5, 3072])


### 6.4. attentions

In [76]:
# Each decoder layer returns its attention weights.
# Shape of each attention tensor:
# [batch_size, num_heads, sequence_length, sequence_length]
# For a sequence of length 5, attention maps are 5x5 matrices for each head.

print("Attention Weights — Last Decoder Layer, Last Attention Head")

# Get the attention map from:
# - The final decoder layer (index -1 or 31)
# - The first (and only) sequence in the batch (index 0)
# - The last attention head (index -1 or 31)
attn = model_output.attentions[-1][0][-1]  # Shape: [seq_len, seq_len]
print(attn)

# For reference:
print("\nTotal Decoder Layers with Attention:", len(model_output.attentions))

# Check again using full indexing syntax for clarity
attn_debug = model_output.attentions[31][0][31]
print("\nAttention Matrix [Layer 31][Batch 0][Head 31]:\n", attn_debug)

# 🧠 Interpretation:
# - This is a softmax-normalized matrix: each row sums to ~1.0
# - It shows where each token is "looking" (attending) in the sequence.
# - The causal (look-ahead) mask ensures that token i can only attend to tokens 0...i.
#   → Hence the upper triangle is all zeros.

Attention Weights — Last Decoder Layer, Last Attention Head
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.9426, 0.0574, 0.0000, 0.0000, 0.0000],
        [0.3606, 0.0461, 0.5933, 0.0000, 0.0000],
        [0.7335, 0.1014, 0.0246, 0.1405, 0.0000],
        [0.0928, 0.0285, 0.0204, 0.8339, 0.0244]], device='mps:0',
       grad_fn=<SelectBackward0>)

Total Decoder Layers with Attention: 32

Attention Matrix [Layer 31][Batch 0][Head 31]:
 tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.9426, 0.0574, 0.0000, 0.0000, 0.0000],
        [0.3606, 0.0461, 0.5933, 0.0000, 0.0000],
        [0.7335, 0.1014, 0.0246, 0.1405, 0.0000],
        [0.0928, 0.0285, 0.0204, 0.8339, 0.0244]], device='mps:0',
       grad_fn=<SelectBackward0>)


In [78]:
# We'll convert attention weights to a DataFrame for easier inspection.
# This helps us understand which words each token is attending to.

import pandas as pd

# 🔍 Get attention matrix from:
# - Layer 31 (last layer)
# - Batch 0 (only one)
# - Head 31 (last head)
tensor = model_output.attentions[31][0][31]  # Shape: [seq_len, seq_len]

# Convert to a DataFrame: rows = query tokens, columns = keys
df = pd.DataFrame(tensor.detach().cpu().numpy(), columns=prompt.split(), index=prompt.split())

print("\nAttention weights from the last head of the last decoder layer:")
display(df)

# Now let's inspect the average attention across all heads in the last layer

# Average across heads → [num_heads, seq_len, seq_len] → mean over dim=0
last_layer_attn = model_output.attentions[-1][0]  # shape: [32 heads, seq_len, seq_len]
avg_attn = last_layer_attn.mean(dim=0)  # shape: [seq_len, seq_len]

# Convert to DataFrame for visualization
df_avg = pd.DataFrame(avg_attn.detach().cpu().numpy(), columns=prompt.split(), index=prompt.split())

print("\nAverage Attention (all heads in last layer):")
display(df_avg)

# Notes:
# - These matrices show **how strongly each token attends to others in the sequence**.
# - Rows = query tokens (trying to "understand")
# - Columns = key tokens (tokens being attended to)
# - For example, the row for "is" might show strong weights on "Germany" and "capital".

# Interpretation Tip:
# The model's final layer and head think that "The" is most important when predicting "is" — 
# but this is just one slice of the reasoning process.
# Earlier layers or heads may have captured higher-level semantics (like phrase structure or world knowledge).


Attention weights from the last head of the last decoder layer:


,The,capital,of,Germany,is
The,1.000000,0.000000,0.000000,0.000000,0.000000
capital,0.942646,0.057354,0.000000,0.000000,0.000000
of,0.360588,0.046102,0.593311,0.000000,0.000000
Germany,0.733494,0.101438,0.024600,0.140468,0.000000
is,0.092815,0.028465,0.020437,0.833856,0.024428



Average Attention (all heads in last layer):


,The,capital,of,Germany,is
The,1.000000,0.000000,0.000000,0.000000,0.000000
capital,0.767401,0.232599,0.000000,0.000000,0.000000
of,0.722282,0.086685,0.191034,0.000000,0.000000
Germany,0.677479,0.062307,0.062907,0.197307,0.000000
is,0.593239,0.045724,0.031294,0.181116,0.148627


## 7. From Hidden States to Vocabulary Logits (LM Head)

In [79]:
# Step 1: Extract the final hidden states from the model output
# Shape: [batch_size=1, sequence_length=5, hidden_dim=3072]
last_hid_state = model_output.last_hidden_state

# Step 2: Pass hidden states through the LM head
# This linear layer maps from hidden_dim (3072) → vocab_size (32064)
# Output shape: [batch_size=1, sequence_length=5, vocab_size=32064]
lm_head_output = model.lm_head(last_hid_state)

print("Shape of Logits (LM Head Output):", lm_head_output.shape)

# Each position in the sequence has a full vocabulary-size vector of logits
# These represent the model's confidence scores for every possible next token

# During text generation:
# - We only use the **last token's output** to predict the next token.
# - This is because generation is autoregressive: one token at a time.
last_token_logits = lm_head_output[0, -1]  # Shape: [vocab_size]
print("\nLogits for Last Token in Sequence:")
print(last_token_logits)

# Step 3: Decode the most likely next token using greedy decoding (argmax)
pred_token_id = last_token_logits.argmax(-1)  # ID of the highest scoring token
pred_token = tokenizer.decode(pred_token_id)

print("\nPredicted Next Token:", pred_token)

# Optional: Inspect logits for all positions (like in training)
# Even though we use only the last one during generation, the model outputs logits for all tokens:
print("\nTop predictions for each position in the sequence:")
for i in range(len(input_ids[0])):
    token_id = lm_head_output[0, i].argmax(-1)
    print(f"Token {i} prediction → {tokenizer.decode(token_id)}")

Shape of Logits (LM Head Output): torch.Size([1, 5, 32064])

Logits for Last Token in Sequence:
tensor([26.8628, 28.4439, 27.8377,  ..., 20.2959, 20.3005, 20.2998],
       device='mps:0', grad_fn=<SelectBackward0>)

Predicted Next Token: Berlin

Top predictions for each position in the sequence:
Token 0 prediction → code
Token 1 prediction → of
Token 2 prediction → the
Token 3 prediction → is
Token 4 prediction → Berlin
